# Product-Image Classifier (CNN) — Apparel vs Electronics vs Home

**Goal.** Flipkart-style catalogue auto-categorisation: train a small CNN on real
e-commerce product images, push **validation accuracy past 85%**, then show exactly
**which category pairs the model still fumbles — and why**.

**Data.** Public [Amazon Reviews 2023](https://huggingface.co/datasets/McAuley-Lab/Amazon-Reviews-2023)
corpus (Hou et al., 2024): `Amazon Fashion → Apparel`, `Electronics → Electronics`,
`Home & Kitchen → Home`. 12,469 real product photos, balanced classes, 70/15/15 split.

**Models.**
1. `ProductCNN` — small from-scratch CNN (0.63M params). Honest baseline; underfits noisy real-world photos.
2. `MobileNetV3-Small` (1.5M params, ImageNet-pretrained) fine-tuned — **final classifier, >85% val accuracy**.

> **Runtime → Change runtime type → T4 GPU.** ~8 min total on GPU.

In [ ]:
# 1 ▸ Setup
import torch, torchvision
print("torch", torch.__version__, "| cuda:", torch.cuda.is_available())
import numpy as np, pandas as pd, matplotlib.pyplot as plt
device = "cuda" if torch.cuda.is_available() else "cpu

In [ ]:
# 2 ▸ Get the dataset bundle (images + splits + metadata)
#     Released as an asset of the GitHub repo (12.5k real product images, 256px).
import os, urllib.request
REPO = "akash-pathak-scientist/product-image-classifier-cnn"
if not os.path.exists("dataset.zip"):
    urllib.request.urlretrieve(
        f"https://github.com/{REPO}/releases/latest/download/dataset.zip",
        "dataset.zip")
!unzip -q -o dataset.zip
import glob; print("images:", len(glob.glob("data/raw/*/*.jpg")))
!head -2 data/splits/train.csv

In [ ]:
# 3 ▸ Datasets & loaders
import torchvision.transforms as T
from torch.utils.data import DataLoader
from PIL import Image

CLASSES = ["Apparel", "Electronics", "Home"]
MEAN, STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def make_train_tf(size):
    return T.Compose([
        T.RandomResizedCrop(size, scale=(0.75, 1.0)),
        T.RandomHorizontalFlip(), T.RandomRotation(8),
        T.ColorJitter(0.2, 0.2, 0.2),
        T.ToTensor(), T.Normalize(MEAN, STD)])

def make_eval_tf(size):
    return T.Compose([T.Resize((size, size)), T.ToTensor(), T.Normalize(MEAN, STD)])

class ProductDS(torch.utils.data.Dataset):
    def __init__(self, csv, tf):
        self.df = pd.read_csv(csv); self.tf = tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        return self.tf(Image.open("data/raw/" + r.file).convert("RGB")), \
               CLASSES.index(r.label)

SIZE = 224  # MobileNetV3-S input (final protocol)
tr = DataLoader(ProductDS("data/splits/train.csv", make_train_tf(SIZE)), 64,
                shuffle=True, num_workers=2, pin_memory=True, drop_last=True)
va = DataLoader(ProductDS("data/splits/val.csv", make_eval_tf(SIZE)), 256,
                num_workers=2, pin_memory=True)
te = DataLoader(ProductDS("data/splits/test.csv", make_eval_tf(SIZE)), 256,
                num_workers=2)
len(tr.dataset), len(va.dataset), len(te.dataset)

In [ ]:
# 4 ▸ Models
# (A) ProductCNN — from scratch, 0.63M params (baseline experiment)
import torch.nn as nn

class ConvBlock(nn.Sequential):
    def __init__(self, cin, cout):
        super().__init__(nn.Conv2d(cin, cout, 3, padding=1, bias=False),
                         nn.BatchNorm2d(cout), nn.ReLU(True), nn.MaxPool2d(2))

class ProductCNN(nn.Module):
    def __init__(self, num_classes=3, dropout=0.30):
        super().__init__()
        chs, cin = [32, 64, 128, 256], 3
        blocks = []
        for c in chs:
            blocks.append(ConvBlock(cin, c)); cin = c
        self.features = nn.Sequential(*blocks)
        self.head = nn.Sequential(nn.AdaptiveAvgPool2d(1), nn.Flatten(),
                                  nn.Linear(chs[-1], 128), nn.ReLU(True),
                                  nn.Dropout(dropout), nn.Linear(128, num_classes))
    def forward(self, x): return self.head(self.features(x))

# (B) FINAL: MobileNetV3-Small, ImageNet-pretrained, fine-tuned head+trunk
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

def build_final(num_classes=3, dropout=0.2):
    m = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1)
    m.classifier[2] = nn.Dropout(dropout)
    m.classifier[3] = nn.Linear(m.classifier[3].in_features, num_classes)
    return m

model = build_final().to(device)
sum(p.numel() for p in model.parameters())/1e6, "M parameters (final model)"

In [ ]:
# 5 ▸ Train (final model) — AdamW, cosine schedule, label smoothing, early stop
EPOCHS, PATIENCE, TARGET = 12, 5, 0.85
crit = nn.CrossEntropyLoss(label_smoothing=0.05)
opt = torch.optim.AdamW([{"params": [p for n,p in model.named_parameters()
                                     if not n.startswith("classifier")], "lr": 8e-5},
                         {"params": [p for n,p in model.named_parameters()
                                     if n.startswith("classifier")], "lr": 4e-4}],
                        weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=2e-5)

def run(loader, train=False):
    model.train() if train else model.eval()
    L, C_, N = 0., 0, 0
    with torch.set_grad_enabled(train):
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            out = model(x); loss = crit(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            L += loss.item()*len(y); C_ += (out.argmax(1)==y).sum().item(); N += len(y)
    return L/N, C_/N

hist, best, since = [], 0., 0
for ep in range(1, EPOCHS+1):
    tl, ta = run(tr, True); vl, v_acc = run(va)
    sched.step()
    hist.append((ep, tl, ta, vl, v_acc))
    print(f"ep {ep:02d} | train {ta:.4f} | val {v_acc:.4f}")
    if v_acc > best: best, since = v_acc, 0; torch.save(model.state_dict(), "best.pt")
    else: since += 1
    if (best >= TARGET and since >= 3) or since >= PATIENCE:
        print("stop:", f"best val {best:.4f}"); break
model.load_state_dict(torch.load("best.pt"))
print("BEST val_acc:", round(best, 4))

In [ ]:
# 6 ▸ (optional) Baseline: train ProductCNN from scratch with the same loop
#     Expected: it plateaus well below the transfer model -> motivates transfer.
scratch = ProductCNN().to(device)
model, best_bak = scratch, None
opt = torch.optim.AdamW(model.parameters(), lr=1.5e-3, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=12, eta_min=1e-4)
for ep in range(1, 13):
    tl, ta = run(tr, True); vl, v_acc = run(va)
    print(f"scratch ep {ep:02d} | train {ta:.4f} | val {v_acc:.4f}")
model = scratch  # keep weights; final predictions below use the transfer model
# restore the transfer model:
model = build_final().to(device); model.load_state_dict(torch.load("best.pt"))

In [ ]:
# 7 ▸ Test-set evaluation + confusion matrix (final model)
from sklearn.metrics import classification_report, confusion_matrix

ys, ps = [], []
model.eval()
with torch.no_grad():
    for x, y in te:
        ps += model(x.to(device)).argmax(1).cpu().tolist(); ys += y.tolist()
ys, ps = np.array(ys), np.array(ps)
acc = (ys == ps).mean()
print(f"TEST ACCURACY: {acc:.4f}  (n={len(ys)})\n")
print(classification_report(ys, ps, target_names=CLASSES, digits=4))

cm = confusion_matrix(ys, ps)
fig, axes = plt.subplots(1, 2, figsize=(11, 4.6))
for ax, mat, fmt, ttl in [(axes[0], cm, "d", "counts"),
                          (axes[1], cm/cm.sum(1, keepdims=True), ".2%", "row-normalised")]:
    ax.imshow(mat, cmap="Blues")
    ax.set_xticks(range(3), CLASSES, rotation=20); ax.set_yticks(range(3), CLASSES)
    ax.set_xlabel("Predicted"); ax.set_ylabel("True"); ax.set_title(ttl)
    for i in range(3):
        for j in range(3):
            ax.text(j, i, format(mat[i, j], fmt), ha="center", va="center",
                    color="white" if mat[i, j] > mat.max()*0.6 else "black")
fig.suptitle(f"Final model — test accuracy {acc:.2%}"); fig.tight_layout()

In [ ]:
# 8 ▸ Which pairs fumble — and which products cause it?
test_df = pd.read_csv("data/splits/test.csv"); test_df["pred"] = [CLASSES[i] for i in ps]
test_df["asin"] = test_df.file.str.split("/").str[-1].str.replace(".jpg", "", regex=False)
meta = pd.read_csv("data/metadata.csv")            # ships in dataset.zip
m = test_df.merge(meta[["asin", "subcategory", "title"]], on="asin", how="left")
wrong = m[m.label != m.pred]
support = m.groupby("label").size()
print("Misclassified volume per pair:")
for (t, p), n in wrong.groupby(["label", "pred"]).size().sort_values(ascending=False).items():
    print(f"  {t:<12} -> {p:<12} {n:>4}  ({n/support[t]:.1%} of true {t})")
print()
for (t, p), g in wrong.groupby(["label", "pred"]):
    print(f"--- {t} -> {p}: top sub-categories ---")
    print(g.subcategory.value_counts().head(5).to_string(), "\n")

In [ ]:
# 9 ▸ Training curves
h = pd.DataFrame(hist, columns=["ep", "tl", "ta", "vl", "va"])
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(h.ep, h.tl, label="train"); ax[0].plot(h.ep, h.vl, label="val")
ax[0].set_title("Loss"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(h.ep, h.ta, label="train"); ax[1].plot(h.ep, h.va, label="val")
ax[1].axhline(.85, ls="--", c="red"); ax[1].text(h.ep.iloc[0], .852, "target 85%", c="red")
ax[1].set_title("Accuracy"); ax[1].legend(); ax[1].grid(alpha=.3)